### Chapter 3.1 - Linear Regression

Linear regression is the simplest supervised learning model for predicting a number from input features. This notebook builds it from plain weighted sums before connecting it to vectorized tensor code and neural network language.


In [ ]:
import math
import random
import numpy as np
import torch


### 3.1.1 Basics

#### 1. Intuition

Regression means predicting a numerical value. Linear regression predicts that value by adding weighted input features plus a bias.

A feature is an input measurement. A label is the target value. A weight says how strongly one feature affects the prediction. A bias is a learned offset added after the weighted features.

For one house, a tiny model might say: predicted price = size weight times size + bedroom weight times bedrooms + bias.

#### 2. Why this exists

Linear regression gives us the first complete learning problem with inputs, labels, parameters, predictions, and loss.

It is simple enough to compute by hand, but it already has the same structure as larger neural networks: choose parameters, make predictions, measure errors, and adjust parameters.


#### 3. Examples

Manual Python: compute one prediction from two features.


In [ ]:
features = [2.0, 3.0]
weights = [4.0, 1.0]
bias = 5.0
prediction = features[0] * weights[0] + features[1] * weights[1] + bias
prediction


NumPy: the same weighted sum as a dot product.


In [ ]:
x_np = np.array([2.0, 3.0])
w_np = np.array([4.0, 1.0])
b_np = 5.0
prediction_np = np.dot(x_np, w_np) + b_np
prediction_np


PyTorch: the same idea using tensors.


In [ ]:
x = torch.tensor([2.0, 3.0])
w = torch.tensor([4.0, 1.0])
b = torch.tensor(5.0)
prediction = torch.dot(x, w) + b
prediction


#### 4. Step-by-step breakdown

The manual expression multiplies each feature by its matching weight.

The products are added together, then the bias is added.

`np.dot(x_np, w_np)` performs the multiply-then-sum pattern for vectors.

`torch.dot(x, w)` does the same thing in PyTorch for 1-dimensional tensors.

At this stage the weights and bias are just chosen numbers. Training will later mean changing those numbers so predictions improve.

#### 5. Connection to ML systems

A neural network layer can be viewed as a repeated weighted-sum operation. Linear regression is therefore the smallest useful bridge between ordinary algebra and neural network code.

#### 6. Common confusion points

- Linear does not mean easy; it means the prediction is a weighted sum of inputs.
- The bias is not attached to one feature. It shifts the whole prediction.
- A feature vector and weight vector must have matching length.
- Prediction is not learning. Learning starts when parameters are adjusted from data.


### 3.1.2 Vectorization for Speed

#### 1. Intuition

Vectorization means replacing many explicit Python loops with array or tensor operations.

A batch is a small group of examples processed together. If each row is one example, a batch of examples forms a matrix.

#### 2. Why this exists

Python loops are slow for large numerical work. Tensor libraries send the repeated arithmetic to optimized low-level code and hardware.

Vectorization also makes the mathematical intent clearer: one matrix of inputs times one vector of weights gives many predictions.


#### 3. Examples

Manual Python: predict two examples with a loop.


In [ ]:
X = [[2.0, 3.0], [1.0, 4.0]]
w = [4.0, 1.0]
b = 5.0
preds = []
for x in X:
    preds.append(x[0] * w[0] + x[1] * w[1] + b)
preds


PyTorch: predict the batch with one matrix-vector product.


In [ ]:
X = torch.tensor([[2.0, 3.0], [1.0, 4.0]])
w = torch.tensor([4.0, 1.0])
b = torch.tensor(5.0)
preds = X @ w + b
preds


#### 4. Step-by-step breakdown

The manual loop handles one row at a time.

Each row is a feature vector for one example.

`X @ w` multiplies the input matrix by the weight vector. Each output value is one row's dot product with `w`.

`+ b` uses broadcasting to add the same scalar bias to every prediction.

#### 5. Connection to ML systems

Training loops usually process minibatches. A minibatch is a batch used for one parameter update. Vectorization makes minibatch training practical.

#### 6. Common confusion points

- Vectorization changes how computation is expressed, not the math being computed.
- `X @ w` is matrix-vector multiplication, not elementwise multiplication.
- Batch rows must use the same feature order.
- Broadcasting the bias is intentional here, but broadcasting mistakes can hide shape bugs.


### 3.1.3 The Normal Distribution and Squared Loss

#### 1. Intuition

A loss is a number measuring how wrong a prediction is. Squared loss uses the squared difference between prediction and label.

Noise means variation we do not fully explain with features. The normal distribution is a bell-shaped probability distribution often used to model small random errors around zero.

#### 2. Why this exists

Squared loss punishes large errors more than small errors because errors are squared.

If we assume labels equal a linear prediction plus normal noise, minimizing squared loss is a natural training objective.


#### 3. Examples

Manual Python: compute squared errors and their mean.


In [ ]:
preds = [10.0, 12.0, 15.0]
labels = [11.0, 10.0, 15.0]
errors = [preds[i] - labels[i] for i in range(3)]
squared = [e * e for e in errors]
mean_loss = sum(squared) / len(squared)
mean_loss


PyTorch: the same loss with tensors.


In [ ]:
preds = torch.tensor([10.0, 12.0, 15.0])
labels = torch.tensor([11.0, 10.0, 15.0])
losses = (preds - labels) ** 2
loss = losses.mean()
loss


#### 4. Step-by-step breakdown

`preds[i] - labels[i]` computes one prediction error.

Squaring makes negative and positive errors both positive.

The mean turns per-example losses into one scalar loss.

In PyTorch, `preds - labels` computes all errors elementwise. `** 2` squares each error. `.mean()` reduces the vector into one scalar.

#### 5. Connection to ML systems

Most training code needs one scalar loss before calling `backward()`. Squared loss is the standard first regression loss because it is simple, smooth, and easy to differentiate.

#### 6. Common confusion points

- Squared loss is not the same as absolute error.
- Squaring makes large errors disproportionately expensive.
- A small mean loss can hide one very bad example.
- The normal-noise story is a modeling assumption, not a guarantee about real data.


### 3.1.4 Linear Regression as a Neural Network

#### 1. Intuition

A neural network is a parameterized function built from layers. A layer is a reusable computation block.

Linear regression can be seen as a neural network with one linear layer and no hidden layers. A hidden layer is an intermediate layer between inputs and final outputs.

#### 2. Why this exists

This viewpoint matters because it shows that neural networks are not a separate universe. They generalize the same weighted-sum pattern.

Understanding linear regression as a layer makes later PyTorch modules easier to read.


#### 3. Examples

PyTorch layer version: one linear layer maps two inputs to one output.


In [ ]:
layer = torch.nn.Linear(2, 1)
X = torch.tensor([[2.0, 3.0]])
y_hat = layer(X)
y_hat.shape


Inspect the parameters stored by the layer.


In [ ]:
weight_shape = layer.weight.shape
bias_shape = layer.bias.shape
weight_shape, bias_shape


#### 4. Step-by-step breakdown

`torch.nn.Linear(2, 1)` creates a layer with 2 input features and 1 output value.

`layer(X)` calls the layer on input data. The layer computes a weighted sum plus bias internally.

`layer.weight` stores the learnable weights. `layer.bias` stores the learnable bias.

The initial values are random or default-initialized because training has not happened yet.

#### 5. Connection to ML systems

Later models will stack many layers. The first step is recognizing that even a tiny layer contains parameters and a forward computation.

#### 6. Common confusion points

- A layer object stores parameters; it is not just a formula written on paper.
- Calling `layer(X)` runs the layer's forward computation.
- The layer's output is not meaningful before training.
- Linear regression has no hidden layer, but it can still be described with neural network language.


### 3.1.5 Summary

#### 1. Intuition

Linear regression predicts numbers with weighted sums.

The core objects are features, labels, weights, bias, predictions, and loss.

#### 2. Why this exists

This chapter gives the smallest complete supervised learning system. Every later training example reuses these pieces with more complex models or data.


#### 3. Examples

A compact linear regression prediction and loss.


In [ ]:
X = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
w = torch.tensor([2.0, -1.0])
b = torch.tensor(0.5)
y = torch.tensor([0.0, 2.0])
y_hat = X @ w + b
loss = ((y_hat - y) ** 2).mean()
loss


#### 4. Step-by-step breakdown

`X @ w + b` computes one prediction per row.

`y_hat - y` computes prediction errors.

Squaring and averaging create a scalar loss.

The code is small, but it contains the basic structure of model training.

#### 5. Connection to ML systems

A training system repeatedly computes this kind of loss, asks automatic differentiation for gradients, and updates parameters.

#### 6. Common confusion points

- Linear regression is a full learning problem, not just a formula.
- Shape tracking is part of correctness.
- Loss defines what the model is trying to improve.
- Parameters are useful only after they are fit to data.


### 3.1.6 Exercises

#### 1. Intuition

Exercises should make you predict shapes, values, and loss behavior before relying on the framework.

#### 2. Why this exists

Linear regression is simple enough that every number can be inspected. That makes it ideal for debugging your mental model.


#### 3. Examples

Exercise 1: compute predictions for three examples.


In [ ]:
X = torch.tensor([[1.0, 0.0], [0.0, 1.0], [2.0, 2.0]])
w = torch.tensor([3.0, 4.0])
b = torch.tensor(1.0)
y_hat = X @ w + b
y_hat


Exercise 2: compute mean squared loss.


In [ ]:
y = torch.tensor([4.0, 5.0, 15.0])
loss = ((y_hat - y) ** 2).mean()
loss


#### 4. Step-by-step breakdown

Exercise 1 checks matrix-vector multiplication and bias broadcasting.

Exercise 2 checks the squared-loss formula.

If you can compute the first prediction by hand, the tensor version becomes less mysterious.

#### 5. Connection to ML systems

These exact pieces appear in later from-scratch training loops and concise PyTorch implementations.

#### 6. Common confusion points

- Check feature order before interpreting a weight.
- Check prediction shape before computing loss.
- Squared loss returns a scalar only after reduction.
- A low loss for one tiny batch does not prove generalization.
